<a href="https://colab.research.google.com/github/vignesh-potharaj/gen-ai/blob/main/Legal_Assistaint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =====================================================================
# TASK 1 & 2: Document Processing, Chunking, and Metadata Extraction
# =====================================================================

import re
from typing import List, Dict, Any

# Mock database of raw, lengthy legal case documents for demonstration
LEGAL_DOCUMENTS = [
    {
        "id": "case_001",
        "text": """SUPREME COURT OF THE UNITED STATES. State of Rajasthan v. Vikram Air Ltd. Year: 2023. Jurisdiction: Federal.
        The court analyzed the applicability of environmental compliance mandates under Section 42 of the Clean Air Enactment.
        It was argued by the petitioner that the appellate tribunal erred in interpreting the timeline restrictions for emissions compliance.
        The respondent maintained that the operational delay constituted a material breach of public trust.
        Ultimately, the bench held that statutory environmental protocols supersede private commercial interests, establishing a strict liability framework for industrial oversight."""
    },
    {
        "id": "case_002",
        "text": """HIGH COURT OF JUDICATURE. Smith v. Digital Infrastructure Corp. Year: 2024. Jurisdiction: State.
        This matter concerns corporate data negligence and individual privacy rights under the Information Security Act.
        The plaintiff asserted that the corporate entity failed to implement reasonable cryptographic security controls, resulting in an unauthorized breach of user records.
        The defense raised a jurisdictional objection, claiming the data servers resided outside the territorial mandate of this court.
        The presiding judge rejected the jurisdictional plea, ruling that systemic consumer interactions within the state subject the entity to its judicial oversight. The court awarded compensatory damages to the affected class."""
    }
]

def chunk_and_metadata_processor(
    documents: List[Dict[str, str]],
    chunk_size: int = 150,
    overlap: int = 30
) -> List[Dict[str, Any]]:
    """
    Splits long legal documents into smaller overlapping text chunks
    and extracts key metadata properties (Case Name, Year, Jurisdiction).
    """
    processed_chunks = []

    for doc in documents:
        text = doc["text"]

        # Metadata Extraction via Simple Regex Parsing
        case_name_match = re.search(r"(?:STATES\.|JUDICATURE\.)\s*(.*?)\.\s*Year:", text)
        year_match = re.search(r"Year:\s*(\d{4})", text)
        jurisdiction_match = re.search(r"Jurisdiction:\s*(\w+)", text)

        metadata = {
            "doc_id": doc["id"],
            "case_name": case_name_match.group(1).strip() if case_name_match else "Unknown Case",
            "year": year_match.group(1).strip() if year_match else "Unknown Year",
            "jurisdiction": jurisdiction_match.group(1).strip() if jurisdiction_match else "Unknown Jurisdiction"
        }

        # Sliding Window Chunking Logic
        words = text.split()
        i = 0
        chunk_idx = 0

        while i < len(words):
            # Formulate the chunk text from a subset of words
            chunk_words = words[i:i + chunk_size]
            chunk_text = " ".join(chunk_words)

            # Create a distinct chunk record containing its metadata context
            chunk_record = {
                "chunk_id": f"{doc['id']}_chunk_{chunk_idx}",
                "text": chunk_text,
                "metadata": metadata
            }
            processed_chunks.append(chunk_record)

            # Increment pointers by chunk size minus overlap
            i += (chunk_size - overlap)
            chunk_idx += 1

            # Prevent infinite loops if configuration parameters are malformed
            if chunk_size <= overlap:
                break

    return processed_chunks

# Execute processing pipeline
processed_chunks = chunk_and_metadata_processor(LEGAL_DOCUMENTS)

# Display sample output chunk structures
print(f"Total processed chunks generated: {len(processed_chunks)}\n")
for chunk in processed_chunks:
    print(f"ID: {chunk['chunk_id']}")
    print(f"Metadata: {chunk['metadata']}")
    print(f"Text Preview: {chunk['text'][:120]}...\n" + "-"*40)

Total processed chunks generated: 2

ID: case_001_chunk_0
Metadata: {'doc_id': 'case_001', 'case_name': 'State of Rajasthan v. Vikram Air Ltd', 'year': '2023', 'jurisdiction': 'Federal'}
Text Preview: SUPREME COURT OF THE UNITED STATES. State of Rajasthan v. Vikram Air Ltd. Year: 2023. Jurisdiction: Federal. The court a...
----------------------------------------
ID: case_002_chunk_0
Metadata: {'doc_id': 'case_002', 'case_name': 'Smith v. Digital Infrastructure Corp', 'year': '2024', 'jurisdiction': 'State'}
Text Preview: HIGH COURT OF JUDICATURE. Smith v. Digital Infrastructure Corp. Year: 2024. Jurisdiction: State. This matter concerns co...
----------------------------------------


In [ ]:
# =====================================================================
# STEP 0: Dependencies Installation and Environment Setup
# =====================================================================

!pip install -q -U google-genai faiss-cpu sentence-transformers

print("✅ Dependencies installed successfully!")
print("• google-genai: The official, updated SDK for Gemini 2.5 Flash.")
print("• faiss-cpu: Facebook AI Similarity Search for efficient vector retrieval.")
# Note: sentence-transformers is installed as a local, free fallback
# to generate text embeddings for our vector store index.

In [2]:
# =====================================================================
# TASK 3 & 4: Embedding Generation, FAISS Indexing, and Retrieval
# =====================================================================

import numpy as np
import faiss as faiss
from sentence_transformers import SentenceTransformer

# 1. Initialize a lightweight, high-performance embedding model
# This maps our semantic legal text into a 384-dimensional dense vector space.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Extract texts from our previously defined processed chunks
chunk_texts = [chunk["text"] for chunk in processed_chunks]

# 3. Generate dense vector embeddings for all legal text chunks
print("Generating embeddings for text chunks...")
chunk_embeddings = embedding_model.encode(chunk_texts, show_progress_bar=False)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

# 4. Construct a FAISS Index for inner product (cosine similarity if normalized)
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# Normalize the vectors to use Inner Product (IndexFlatIP) as Cosine Similarity
faiss.normalize_L2(chunk_embeddings)
index.add(chunk_embeddings)
print(f"Successfully indexed {index.ntotal} chunks into FAISS.\n")

def retrieve_and_rank_cases(query: str, top_k: int = 2) -> List[Dict[str, Any]]:
    """
    Encodes a user query, searches the FAISS index, and returns the
    top_k most relevant chunks along with their metadata and scores.
    """
    # Encode and normalize the user search query
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")
    faiss.normalize_L2(query_embedding)

    # Execute vector search within the FAISS index
    scores, indices = index.search(query_embedding, top_k)

    retrieved_results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        if idx != -1:  # Ensure a valid index was matched
            matched_chunk = processed_chunks[idx]
            retrieved_results.append({
                "rank": rank,
                "score": float(score),
                "text": matched_chunk["text"],
                "metadata": matched_chunk["metadata"]
            })

    return retrieved_results

# --- Test the Retrieval Framework ---
sample_query = "What did the court rule regarding environmental compliance and industrial oversight?"
retrieved_documents = retrieve_and_rank_cases(sample_query, top_k=1)

print(f"🔍 User Query: '{sample_query}'\n")
print(f"Top Retrieved Match (Score: {retrieved_documents[0]['score']:.4f}):")
print(f"• Case: {retrieved_documents[0]['metadata']['case_name']}")
print(f"• Jurisdiction: {retrieved_documents[0]['metadata']['jurisdiction']}")
print(f"• Text: {retrieved_documents[0]['text']}")

ModuleNotFoundError: No module named 'faiss'